## Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

In [3]:
import os
from dotenv import load_dotenv
import glob

from langchain_classic import text_splitter
from openai import OpenAI

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import login

### Connecting to OpenAI API and Hugging Face:

In [4]:
MODEL = 'gpt-4.1-nano'
db_name = 'vector_db'

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv('HF_TOKEN')

openai = OpenAI()
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### 1. Dividing Documents into Chunks:

In [6]:
# Loading in Everything in The Knowledge Base Using Langchain's Loaders:

folders = glob.glob('knowledge-base/*')
documents = []

for folder in folders:

    # Setting up Document-Type as Folder Name:
    doc_type = os.path.basename(folder)

    # Directory Loader Instance:
    loader = DirectoryLoader(folder, glob= '**/*.md', loader_cls= TextLoader, loader_kwargs= {'encoding': 'utf-8'})

    # Loading Data:
    folder_docs = loader.load()

    # Adding Everything to One List:
    for doc in folder_docs:
        doc.metadata['doc_type']= doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents')

Loaded 76 documents


In [7]:
# Dividing Documents into Chunks:
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap= 200)
chunks = text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks.')

Divided into 413 chunks.
